# 09 · DeepONet — Learning Operators, Not Just Solutions

**Level:** Advanced
**Prerequisites:** `01`–`07`

Every notebook so far trained a network to represent **one** solution of **one**
problem. Change the initial condition or forcing and you retrain from scratch.
**Operator learning** breaks that limit: it learns the *mapping* from an entire
input function to the corresponding solution function. Once trained, inference on
a new input is a single forward pass — milliseconds instead of a fresh
optimisation.

**DeepONet** (Lu et al. 2021, *Nature Machine Intelligence*) is the seminal
architecture. It has two sub-networks:
- a **branch net** that encodes the input function `u` (sampled at fixed sensors),
- a **trunk net** that encodes the query location `y`,

and combines them by a dot product: `G(u)(y) ≈ Σ_k b_k(u) · t_k(y)`.

### What you will learn
1. The branch/trunk architecture and why it works (universal approximation of operators)
2. Generating a dataset of (input function, output function) pairs
3. Training and testing generalisation to unseen functions

### References
- Lu, Jin, Pang, Zhang & Karniadakis (2021), *Learning nonlinear operators via DeepONet*, Nat. Mach. Intell. 3:218-229
- Wang, Wang & Perdikaris (2021), *Learning the solution operator ... with physics-informed DeepONets*, Sci. Adv. 7:eabi8605


## 1 · The operator we will learn

The **antiderivative** operator:

$$G:\; u(x)\;\longmapsto\; s(x)=\int_0^x u(\tau)\,d\tau,\qquad s(0)=0.$$

This is Lu et al.'s introductory example. Inputs $u$ are random smooth functions;
targets $s$ are their integrals (computed here by cumulative trapezoid). The
network never sees the integral rule — it learns the operator from examples.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

m = 100                       # number of sensors (fixed grid for the input function)
x_grid = np.linspace(0, 1, m)

def random_function(n=1, length_scale=0.2):
    # smooth random functions via a squared-exponential Gaussian process
    X = x_grid.reshape(-1,1)
    K = np.exp(-(X - X.T)**2 / (2*length_scale**2)) + 1e-8*np.eye(m)
    L = np.linalg.cholesky(K)
    return (L @ np.random.randn(m, n)).T          # (n, m)

def antiderivative(u):
    # cumulative trapezoid, s(0)=0
    dx = x_grid[1]-x_grid[0]
    s = np.zeros_like(u)
    s[:,1:] = np.cumsum((u[:,1:]+u[:,:-1])/2*dx, axis=1)
    return s


In [ ]:
# --- Build dataset: (branch input u@sensors, trunk input y) -> s(y) ---
def make_dataset(N):
    U = random_function(N)                       # (N, m)  input functions
    S = antiderivative(U)                        # (N, m)  target functions
    # for each function, pick random query locations y
    Q = 30
    idx = np.random.randint(0, m, size=(N, Q))
    y   = x_grid[idx]                            # (N, Q)
    s_y = np.take_along_axis(S, idx, axis=1)     # (N, Q)
    # flatten into (N*Q) training triples
    branch = np.repeat(U, Q, axis=0)            # (N*Q, m)
    trunk  = y.reshape(-1,1)                     # (N*Q, 1)
    target = s_y.reshape(-1,1)                   # (N*Q, 1)
    return (torch.tensor(branch,dtype=torch.float32,device=device),
            torch.tensor(trunk ,dtype=torch.float32,device=device),
            torch.tensor(target,dtype=torch.float32,device=device))

br_tr, tr_tr, y_tr = make_dataset(1000)
print("training triples:", br_tr.shape[0])


In [ ]:
class DeepONet(nn.Module):
    def __init__(self, m, p=40, width=80):
        super().__init__()
        self.branch = nn.Sequential(nn.Linear(m,width), nn.Tanh(),
                                    nn.Linear(width,width), nn.Tanh(),
                                    nn.Linear(width,p))
        self.trunk  = nn.Sequential(nn.Linear(1,width), nn.Tanh(),
                                    nn.Linear(width,width), nn.Tanh(),
                                    nn.Linear(width,p), nn.Tanh())
        self.b0 = nn.Parameter(torch.zeros(1))
    def forward(self, u_sensors, y):
        b = self.branch(u_sensors)     # (N, p)
        t = self.trunk(y)              # (N, p)
        return torch.sum(b*t, dim=1, keepdim=True) + self.b0   # (N,1)

model = DeepONet(m).to(device)
print("params:", sum(p.numel() for p in model.parameters()))


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 8000
bs = 4096
N = br_tr.shape[0]
hist=[]
for ep in range(EPOCHS):
    perm = torch.randperm(N, device=device)[:bs]
    opt.zero_grad()
    pred = model(br_tr[perm], tr_tr[perm])
    loss = torch.mean((pred - y_tr[perm])**2)
    loss.backward(); opt.step(); hist.append(loss.item())
    if ep % 1500 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e}")
print("done")


In [ ]:
# --- Test on brand-new input functions ---
U_test = random_function(3)
S_test = antiderivative(U_test)
y_full = torch.tensor(x_grid, dtype=torch.float32, device=device).view(-1,1)

fig, ax = plt.subplots(1,3, figsize=(15,4))
for i in range(3):
    br = torch.tensor(np.repeat(U_test[i:i+1], m, axis=0), dtype=torch.float32, device=device)
    with torch.no_grad():
        pred = model(br, y_full).cpu().numpy().ravel()
    ax[i].plot(x_grid, S_test[i], 'k-', lw=2, label='true ∫u')
    ax[i].plot(x_grid, pred, 'r--', lw=2, label='DeepONet')
    ax[i].plot(x_grid, U_test[i], color='gray', alpha=0.5, label='input u')
    err = np.linalg.norm(pred-S_test[i])/np.linalg.norm(S_test[i])
    ax[i].set_title(f'unseen function #{i+1} (L2={err:.2e})'); ax[i].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 2 · Why operator learning is a big deal

- **Amortised cost:** training is expensive once; every subsequent solve is a
  forward pass. For design sweeps, control, uncertainty quantification, or
  real-time digital twins this is transformative.
- **Discretisation flexibility:** the trunk takes a continuous query `y`, so you
  can evaluate the output anywhere, at any resolution.
- **Physics-informed variant:** replace (or augment) the data loss with a PDE
  residual on the *output* function — that is a **physics-informed DeepONet**
  (Wang, Wang & Perdikaris 2021), which needs little or no labelled solution data.

DeepONet and the Fourier Neural Operator (FNO, Li et al. 2021) are the two
dominant operator-learning families; see the Research track and the literature
review for the comparison.

## 3 · Exercises
1. Learn a *nonlinear* operator, e.g. `u ↦ solution of an ODE driven by u`.
2. Turn this into a **physics-informed DeepONet**: add a loss enforcing
   `d/dy G(u)(y) = u(y)` (the defining relation of the antiderivative) and reduce
   the number of labelled targets.
3. Compare branch architectures (CNN vs MLP) when `u` is high-dimensional.

**Next (Research track):** `10_navier_stokes_cylinder`,
`11_physics_informed_deeponet`, `12_architectures_and_training`.
